In [1]:
limma_script = '''
.libPaths(c("/rds/homes/j/jxt554/R/library",
             .libPaths()))
library(limma)

DATA_DIR   <- "/rds/homes/j/jxt554/data"
TABLES_DIR <- "/rds/homes/j/jxt554/tables"

cat("DIFFERENTIAL ABUNDANCE — limma\\n")
cat(strrep("=", 55), "\\n")

run_limma <- function(cohort) {
    cat("\\n---", cohort, "---\\n")

    X    <- as.matrix(read.csv(
        file.path(DATA_DIR,
                   paste0("X_", tolower(cohort),
                           "_limma.npy.csv"))))
    meta <- read.csv(
        file.path(DATA_DIR,
                   paste0("meta_",
                           tolower(cohort),
                           "_final.csv")))

    cat("Matrix:", nrow(X), "x", ncol(X), "\\n")
    cat("C1:", sum(meta$cluster==0),
        "C2:", sum(meta$cluster==1), "\\n")

    cluster <- factor(meta$cluster,
                       levels=c(0,1),
                       labels=c("C1","C2"))
    cluster <- relevel(cluster, ref="C1")
    age     <- scale(meta$age)[,1]
    batch   <- factor(meta$batch)
    sex     <- factor(meta$sex)

    smoke_raw <- meta$smoking_status
    smoke_raw[!smoke_raw %in%
               c("Never","Previous","Current")] <- "Never"
    smoking <- factor(smoke_raw,
                       levels=c("Never",
                                 "Previous",
                                 "Current"))

    expr <- t(X)
    rownames(expr) <- colnames(X)

    # M1: base
    design1 <- model.matrix(
        ~cluster+age+batch)
    fit1 <- eBayes(lmFit(expr, design1))

    # M2: +sex
    design2 <- model.matrix(
        ~cluster+age+batch+sex)
    fit2 <- eBayes(lmFit(expr, design2))

    # M3: full (primary)
    design3 <- model.matrix(
        ~cluster+age+batch+sex+smoking)
    fit3 <- eBayes(lmFit(expr, design3))

    get_results <- function(fit, label) {
        coef_name <- grep("C2",
                           colnames(
                               fit$coefficients),
                           value=TRUE)[1]
        res <- topTable(fit,
                         coef=coef_name,
                         number=Inf,
                         adjust="BH",
                         sort.by="P")
        res$protein <- rownames(res)
        n_up <- sum(res$adj.P.Val < 0.05 &
                     res$logFC > 0, na.rm=TRUE)
        n_dn <- sum(res$adj.P.Val < 0.05 &
                     res$logFC < 0, na.rm=TRUE)
        cat(sprintf("  %s: up=%d dn=%d\\n",
                     label, n_up, n_dn))
        write.csv(res,
                   file.path(TABLES_DIR,
                              paste0(
                                  tolower(cohort),
                                  "_limma_",
                                  label,
                                  ".csv")),
                   row.names=FALSE)
        return(res)
    }

    r1 <- get_results(fit1, "M1")
    r2 <- get_results(fit2, "M2")
    r3 <- get_results(fit3, "M3")

    # Sensitivity overlap M1 vs M3
    sig1 <- r1$protein[r1$adj.P.Val < 0.05]
    sig3 <- r3$protein[r3$adj.P.Val < 0.05]
    core <- intersect(sig1, sig3)
    cat(sprintf("  M1 sig: %d\\n", length(sig1)))
    cat(sprintf("  M3 sig: %d\\n", length(sig3)))
    cat(sprintf("  Core (M1 n M3): %d\\n",
                 length(core)))
    if (length(sig1) > 0) {
        cat(sprintf("  %% retained: %.1f\\n",
                     length(core)/
                     length(sig1)*100))
    }

    cat("\\nTop 10 up in C2 (M3):\\n")
    top_up <- head(r3[r3$logFC > 0,
                       c("protein",
                          "logFC",
                          "adj.P.Val")], 10)
    print(top_up, row.names=FALSE)

    cat("\\nTop 10 up in C1 (M3):\\n")
    top_dn <- head(r3[r3$logFC < 0,
                       c("protein",
                          "logFC",
                          "adj.P.Val")], 10)
    print(top_dn, row.names=FALSE)
}

run_limma("CD")
run_limma("UC")
cat("\\nLIMMA COMPLETE\\n")
'''

with open(
    '/rds/homes/j/jxt554/pipeline/07_limma.R',
    'w') as f:
    f.write(limma_script)
print('Saved: 07_limma.R')

Saved: 07_limma.R


In [2]:
import numpy as np
import pandas as pd
import os

DATA_DIR = '/rds/homes/j/jxt554/data'

prots = pd.read_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    header=None)[0].tolist()

for cohort in ['cd','uc']:
    X    = np.load(
        f'{DATA_DIR}/X_{cohort}_limma.npy')
    meta = pd.read_csv(
        f'{DATA_DIR}/meta_{cohort}_final.csv')

    # Align protein names
    n_prots = min(len(prots), X.shape[1])
    X_df    = pd.DataFrame(
        X[:,:n_prots],
        columns=prots[:n_prots])

    X_df.to_csv(
        f'{DATA_DIR}/'
        f'X_{cohort}_limma.npy.csv',
        index=False)
    print(f'✓ Saved: X_{cohort}_limma.npy.csv '
          f'({X_df.shape})')

print('\nRun in terminal:')
print('module load bear-apps/2022b/live &&'
      ' module load R/4.3.1-foss-2022b &&'
      ' Rscript /rds/homes/j/jxt554/'
      'pipeline/07_limma.R')

✓ Saved: X_cd_limma.npy.csv ((215, 991))
✓ Saved: X_uc_limma.npy.csv ((430, 991))

Run in terminal:
module load bear-apps/2022b/live && module load R/4.3.1-foss-2022b && Rscript /rds/homes/j/jxt554/pipeline/07_limma.R


In [3]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import \
    multipletests
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = '/rds/homes/j/jxt554/data'
TABLES_DIR = '/rds/homes/j/jxt554/tables'

prots = pd.read_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    header=None)[0].tolist()

print('WELCH T-TEST + BH FDR')
print('='*55)

for cohort in ['cd','uc']:
    X    = np.load(
        f'{DATA_DIR}/X_{cohort}_limma.npy')
    meta = pd.read_csv(
        f'{DATA_DIR}/meta_{cohort}_final.csv')

    n_prots = min(len(prots), X.shape[1])
    X       = X[:,:n_prots]
    labels  = meta['cluster'].values

    C1 = X[labels==0]
    C2 = X[labels==1]

    print(f'\n{cohort.upper()}: '
          f'C1 n={len(C1)} C2 n={len(C2)}')

    t_stats = []
    p_vals  = []
    fc_vals = []

    for j in range(X.shape[1]):
        t, p = stats.ttest_ind(
            C1[:,j], C2[:,j],
            equal_var=False)
        fc = C2[:,j].mean() - C1[:,j].mean()
        t_stats.append(t)
        p_vals.append(p)
        fc_vals.append(fc)

    # BH correction
    _, q_vals, _, _ = multipletests(
        p_vals, method='fdr_bh')

    res = pd.DataFrame({
        'protein'    : prots[:n_prots],
        'logFC'      : fc_vals,
        't_stat'     : t_stats,
        'p_value'    : p_vals,
        'adj_p_value': q_vals,
    }).sort_values('adj_p_value')

    n_sig = (res['adj_p_value'] < 0.05).sum()
    n_up  = ((res['adj_p_value'] < 0.05) &
              (res['logFC'] > 0)).sum()
    n_dn  = ((res['adj_p_value'] < 0.05) &
              (res['logFC'] < 0)).sum()

    print(f'Significant (FDR<0.05): {n_sig}')
    print(f'  Up in C2: {n_up}')
    print(f'  Up in C1: {n_dn}')

    print(f'\nTop 10 up in C2:')
    top = res[res['logFC']>0].head(10)
    print(top[['protein','logFC',
                'adj_p_value']].to_string(
        index=False))

    res.to_csv(
        f'{TABLES_DIR}/'
        f'{cohort}_welch_ttest.csv',
        index=False)
    print(f'✓ Saved: {cohort}_welch_ttest.csv')

print('\nWELCH T-TEST COMPLETE')

WELCH T-TEST + BH FDR

CD: C1 n=82 C2 n=133
Significant (FDR<0.05): 646
  Up in C2: 3
  Up in C1: 643

Top 10 up in C2:
 protein    logFC  adj_p_value
   APLP1 0.263635     0.000653
    UMOD 0.329157     0.001795
   RASA1 0.102675     0.030562
  BPIFB1 0.175231     0.135827
    TFRC 0.098900     0.142336
  CLEC4C 0.107015     0.195368
     EPO 0.196344     0.197251
  PM20D1 0.426454     0.209229
NTproBNP 0.258191     0.229539
   CNTN5 0.079337     0.261523
✓ Saved: cd_welch_ttest.csv

UC: C1 n=299 C2 n=131
Significant (FDR<0.05): 628
  Up in C2: 13
  Up in C1: 615

Top 10 up in C2:
protein    logFC  adj_p_value
  APLP1 0.388310 8.337558e-08
   HMBS 0.255616 1.054117e-05
   PKLR 0.341888 2.744757e-05
   HBQ1 0.329340 4.625111e-05
    CA1 0.303566 8.861403e-05
   ARG1 0.252572 1.303504e-04
    LXN 0.343015 5.579459e-04
   AHSP 0.366861 5.957992e-04
  BLVRB 0.299599 9.017526e-04
PLA2G10 0.242702 2.098529e-03
✓ Saved: uc_welch_ttest.csv

WELCH T-TEST COMPLETE


In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score)
from sklearn.metrics import (
    roc_auc_score, classification_report,
    adjusted_rand_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from IPython.display import Image, display
import os

DATA_DIR    = '/rds/homes/j/jxt554/data'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'
SEED        = 42

prots = pd.read_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    header=None)[0].tolist()

print('RANDOM FOREST TRIANGULATION')
print('='*55)

rf_summary = []

for cohort in ['cd','uc']:
    print(f'\n{cohort.upper()}:')

    X      = np.load(
        f'{DATA_DIR}/X_{cohort}_limma.npy')
    meta   = pd.read_csv(
        f'{DATA_DIR}/meta_{cohort}_final.csv')
    labels = meta['cluster'].values
    limma  = pd.read_csv(
        f'{TABLES_DIR}/'
        f'{cohort}_limma_M3.csv')

    n_prots = min(len(prots), X.shape[1])
    X       = X[:,:n_prots]

    print(f'  Matrix: {X.shape}')
    print(f'  C1={( labels==0).sum()} '
          f'C2={(labels==1).sum()}')

    # Train RF
    rf = RandomForestClassifier(
        n_estimators=1000,
        max_features='sqrt',
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1)

    cv  = StratifiedKFold(
        n_splits=5, shuffle=True,
        random_state=SEED)
    auc = cross_val_score(
        rf, X, labels,
        cv=cv, scoring='roc_auc')

    print(f'\n  5-fold CV AUC:')
    for i,a in enumerate(auc):
        print(f'    Fold {i+1}: {a:.4f}')
    print(f'  Mean: {auc.mean():.4f} '
          f'± {auc.std():.4f}')

    # Fit on full data
    rf.fit(X, labels)
    gini = rf.feature_importances_
    idx  = np.argsort(gini)[::-1]

    top30_prots = [prots[i]
                    for i in idx[:30]]
    top30_imp   = gini[idx[:30]]

    # Limma top 50
    limma_top50 = limma.nsmallest(
        50,'adj.P.Val')['protein'].tolist()

    overlap = set(top30_prots) & \
              set(limma_top50)
    print(f'\n  RF top30 ∩ limma top50: '
          f'{len(overlap)}/30')

    print(f'\n  Top 20 RF proteins:')
    print(f'  {"Rank":<5} {"Protein":<14} '
          f'{"Importance":>12} {"limma?":>8}')
    print(f'  {"-"*42}')
    for rank,(p,imp) in enumerate(
            zip(top30_prots[:20],
                top30_imp[:20]),1):
        mark = 'YES' if p in limma_top50 \
               else ''
        print(f'  {rank:<5} {p:<14} '
              f'{imp:>12.4f} {mark:>8}')

    # Save
    imp_df = pd.DataFrame({
        'protein'   : prots[:n_prots],
        'gini_imp'  : gini,
    }).sort_values('gini_imp',ascending=False)
    imp_df.to_csv(
        f'{TABLES_DIR}/'
        f'{cohort}_rf_importance.csv',
        index=False)

    rf_summary.append({
        'cohort'     : cohort.upper(),
        'cv_auc_mean': round(auc.mean(),4),
        'cv_auc_std' : round(auc.std(),4),
        'overlap_30' : len(overlap),
    })

pd.DataFrame(rf_summary).to_csv(
    f'{TABLES_DIR}/rf_summary.csv',
    index=False)
print('\n✓ Saved: rf_summary.csv')
print('\nRF COMPLETE')
print('Next: ORA pathway enrichment')

RANDOM FOREST TRIANGULATION

CD:
  Matrix: (215, 991)
  C1=82 C2=133

  5-fold CV AUC:
    Fold 1: 0.9819
    Fold 2: 0.9955
    Fold 3: 0.9815
    Fold 4: 0.9583
    Fold 5: 0.9977
  Mean: 0.9830 ± 0.0140

  RF top30 ∩ limma top50: 21/30

  Top 20 RF proteins:
  Rank  Protein          Importance   limma?
  ------------------------------------------
  1     NUDT5                0.0312      YES
  2     DAG1                 0.0258      YES
  3     HSPA1A               0.0173      YES
  4     TXNRD1               0.0158      YES
  5     CC2D1A               0.0155      YES
  6     SERPINB1             0.0153      YES
  7     DBI                  0.0153      YES
  8     PLIN3                0.0138         
  9     CRADD                0.0136      YES
  10    PARK7                0.0131         
  11    RWDD1                0.0128      YES
  12    STAMBP               0.0126      YES
  13    F11R                 0.0125      YES
  14    CIAPIN1              0.0122      YES
  15    ABL1      